# Поиск аналогов — ваша часть хакатона
BM25, BGE-M3 и гибрид. Классификатор не требуется. Сначала откройте `analog_review/review.html` и заполните оценки. Полная инструкция — `START_HERE.md`.

In [2]:
from pathlib import Path
import sys, json
PROJECT = Path.cwd()
if not (PROJECT / "support_search").exists():
    PROJECT = Path('/Users/memchik/.codex/.chatgpt-projects/g-p-6aac0ef01b4881918e6dfaec0e5a10af')
sys.path.insert(0, str(PROJECT))
SOURCE = Path("/Users/memchik/Downloads/pochtatech_notebook")
REPORTS = PROJECT if (PROJECT / "analog_review").exists() else PROJECT / "outputs"
LOAD_BGE = False  # True для поиска нового текста через BGE или гибрид


In [3]:
from support_search.focused import load_source, AnalogSearch, BGEEncoder
rows, records, texts, query_vectors, history_vectors, manifest, version = load_source(SOURCE)
encoder = None
if LOAD_BGE:
    import torch
    torch.set_num_threads(2)
    model_path = SOURCE / "model_cache/hub/models--BAAI--bge-m3/snapshots" / manifest["revision"]
    encoder = BGEEncoder(model_path)
engine = AnalogSearch(records, history_vectors, encoder)
print("Готово. Записей:", len(records), "Семантическая модель загружена:", encoder is not None)


Готово. Записей: 1931 Семантическая модель загружена: False


## Новый запрос
Меняйте только следующую ячейку. `bm25` работает без модели; `bge` и `hybrid` требуют `LOAD_BGE=True` выше. Первый нейронный запрос медленнее следующих.

In [5]:
TEXT = "Не приходит СМС с кодом входа в приложение"
METHOD = "hybrid" if LOAD_BGE else "bm25"
response = engine.search(TEXT, method=METHOD)
for hit in response["results"]:
    print("ID:", hit["id"], "Категория:", hit["topic"], "Линия:", hit["support_line"])
    print(hit["original_text"])
    print("Результат:", hit["result"] or "Решение в истории не указано")
    print("Других записей в группе:", len(hit["duplicate_variants"]))
    print()
print(response["timing_ms"])


ID: 38620366 Категория: Проблема с авторизацией/ЭЗП/Бонусами/Доверенностями Линия: (1 линия)
Не могу зарегистрироваться в приложении так как не приходит смс с кодом.что делать?  Григорьева Вале---
Результат: Решение предоставлено
Других записей в группе: 0

ID: 38591654 Категория: Прочее Линия: (1 линия)
ИСО Тема: Вопрос. Не приходит смс с кодом для подтверждения. Зайти в личный кабинет для отслеживания отправления по трек номеру не могу. Забыла пароль. Помогите пожайлуста.  Текст:
Результат: Решение предоставлено
Других записей в группе: 0

ID: 38697024 Категория: Проблема с QR-код(подключение/отключение) Линия: (1 линия)
Тема письма: Услуги для населения/Самостоятельная регистрация отправлений  Текст письма: Тематика: Самостоятельная регистрация отправлений Канал поступления обращения: По телефону Дата поступления обращения: : .. ФИ--- Телефон: + Индекс ОПС:  Электронна--- Суть обращения: проблема куар кодом не приходит код
Результат: Дублирование
Других записей в группе: 1

ID: 3861

## Результаты ручной оценки
После экспорта из HTML задайте путь к скачанному CSV. Пустые оценки не считаются нулевыми. Без ручной разметки победитель неизвестен.

In [7]:
from support_search.evaluation import score_pool, paired_interval
RUNS = REPORTS / "analog_review/runs.json"
JUDGMENTS = REPORTS / "analog_review/judgments.csv"  # замените на judgments_reviewed.csv
scores = score_pool(RUNS, JUDGMENTS)
print(json.dumps(scores["summary"], ensure_ascii=False, indent=2))
print(paired_interval(RUNS, scores, "bm25", "bge"))
print(paired_interval(RUNS, scores, "bge", "hybrid"))
print("Задержки выше — поиск по готовым векторам, без encoder. Полные замеры — analog_benchmark.json.")


{
  "bge": {
    "evaluated_queries": 3,
    "manually_judged_queries": 0,
    "empty_output_queries": 3,
    "total_queries": 40,
    "warm_latency_p50_ms": 0.33400000393157825,
    "warm_latency_p95_ms": 0.8912297911592758
  },
  "bm25": {
    "evaluated_queries": 3,
    "manually_judged_queries": 0,
    "empty_output_queries": 3,
    "total_queries": 40,
    "warm_latency_p50_ms": 1.2951669996255077,
    "warm_latency_p95_ms": 3.6039749997144046
  },
  "hybrid": {
    "evaluated_queries": 3,
    "manually_judged_queries": 0,
    "empty_output_queries": 3,
    "total_queries": 40,
    "warm_latency_p50_ms": 1.3038329998380505,
    "warm_latency_p95_ms": 3.562545496970411
  }
}
{'mean_group_delta_ndcg': 0.0, 'ci95': [0.0, 0.0], 'groups': 3, 'note': 'fixed systems; use independent test after validation weight selection'}
{'mean_group_delta_ndcg': 0.0, 'ci95': [0.0, 0.0], 'groups': 3, 'note': 'fixed systems; use independent test after validation weight selection'}
Задержки выше — поиск 